<a href="https://colab.research.google.com/github/SimoesDev1/AT-Python/blob/main/AT_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

AT- 01

In [12]:
import requests
from bs4 import BeautifulSoup

url = "https://m.imdb.com/chart/top/"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)

print("Status:", response.status_code)

soup = BeautifulSoup(response.text, "html.parser")

items = soup.select("ul.ipc-metadata-list li.ipc-metadata-list-summary-item")

print("Filmes encontrados:", len(items))

titulos = []

for item in items:
    titulo = item.select_one("h3")
    if titulo:
        titulos.append(titulo.text.replace(".", "").strip())

titulos[:10]


Status: 200
Filmes encontrados: 25


['The Shawshank Redemption',
 'The Godfather',
 'The Dark Knight',
 'The Godfather Part II',
 '12 Angry Men',
 'The Lord of the Rings: The Return of the King',
 "Schindler's List",
 'The Lord of the Rings: The Fellowship of the Ring',
 'Pulp Fiction',
 'The Good, the Bad and the Ugly']

AT-02

In [13]:
filmes = []

for item in items:
    titulo_tag = item.select_one("h3")

    # Seleciona todos os spans de metadata (ano, duração, classificação)
    meta_tags = item.select("span.cli-title-metadata-item")

    ano = None
    if meta_tags:
        texto_ano = meta_tags[0].text.strip()
        if texto_ano.isdigit():
            ano = int(texto_ano)

    rating_tag = item.select_one("span.ipc-rating-star--rating")

    titulo = titulo_tag.text.strip() if titulo_tag else "N/A"
    rating = float(rating_tag.text) if rating_tag else None

    filmes.append({
        "title": titulo,
        "year": ano,
        "rating": rating
    })

# Exibir os 5 primeiros
for f in filmes[:5]:
    print(f"{f['title']} ({f['year']}) – Nota: {f['rating']}")


The Shawshank Redemption (1994) – Nota: 9.3
The Godfather (1972) – Nota: 9.2
The Dark Knight (2008) – Nota: 9.1
The Godfather Part II (1974) – Nota: 9.0
12 Angry Men (1957) – Nota: 9.0


AT-03


In [14]:
class TV:
    def __init__(self, title, year):
        self.title = title
        self.year = year

    def __str__(self):
        return f"{self.title} ({self.year})"


AT-04

In [15]:
class Movie(TV):
    def __init__(self, title, year, rating):
        super().__init__(title, year)
        self.rating = rating

    def __str__(self):
        return f"{self.title} ({self.year}) – Nota: {self.rating}"


class Series(TV):
    def __init__(self, title, year, seasons, episodes):
        super().__init__(title, year)
        self.seasons = seasons
        self.episodes = episodes

    def __str__(self):
        return f"{self.title} ({self.year}) – Temporadas: {self.seasons}, Episódios: {self.episodes}"


AT-05

In [16]:
catalog = []

# Criação dos objetos Movie a partir do scraping
for f in filmes:
    catalog.append(Movie(f["title"], f["year"], f["rating"]))

# Criação das séries manualmente
catalog.append(Series("Breaking Bad", 2008, 5, 62))
catalog.append(Series("Game of Thrones", 2011, 8, 73))

# Exibir todos (Deixei 10)
for item in catalog[:10]:
    print(item)


The Shawshank Redemption (1994) – Nota: 9.3
The Godfather (1972) – Nota: 9.2
The Dark Knight (2008) – Nota: 9.1
The Godfather Part II (1974) – Nota: 9.0
12 Angry Men (1957) – Nota: 9.0
The Lord of the Rings: The Return of the King (2003) – Nota: 9.0
Schindler's List (1993) – Nota: 9.0
The Lord of the Rings: The Fellowship of the Ring (2001) – Nota: 8.9
Pulp Fiction (1994) – Nota: 8.8
The Good, the Bad and the Ugly (1966) – Nota: 8.8


AT-06

In [17]:
from sqlalchemy import create_engine, Column, Integer, String, Float
from sqlalchemy.orm import declarative_base, sessionmaker

engine = create_engine("sqlite:///imdb.db")
Base = declarative_base()

class MovieDB(Base):
    __tablename__ = "movies"
    id = Column(Integer, primary_key=True)
    title = Column(String, unique=True)
    year = Column(Integer)
    rating = Column(Float)

class SeriesDB(Base):
    __tablename__ = "series"
    id = Column(Integer, primary_key=True)
    title = Column(String, unique=True)
    year = Column(Integer)
    seasons = Column(Integer)
    episodes = Column(Integer)

Base.metadata.create_all(engine)
Session = sessionmaker(bind=engine)
session = Session()

# Inserir dados
for item in catalog:
    try:
        if isinstance(item, Movie):
            obj = MovieDB(title=item.title, year=item.year, rating=item.rating)
        else:
            obj = SeriesDB(title=item.title, year=item.year,
                           seasons=item.seasons, episodes=item.episodes)

        session.add(obj)
        session.commit()
    except:
        session.rollback()


AT-07

In [18]:
import pandas as pd

try:
    df_movies = pd.read_sql("movies", engine)
    df_series = pd.read_sql("series", engine)

    print(df_movies.head())
    print("-----------------------------------------------")
    print(df_series.head())

except Exception as e:
    print("Erro ao ler o banco:", e)


   id                     title  year  rating
0   1  The Shawshank Redemption  1994     9.3
1   2             The Godfather  1972     9.2
2   3           The Dark Knight  2008     9.1
3   4     The Godfather Part II  1974     9.0
4   5              12 Angry Men  1957     9.0
-----------------------------------------------
   id            title  year  seasons  episodes
0   1     Breaking Bad  2008        5        62
1   2  Game of Thrones  2011        8        73


AT-08

In [23]:
# Ordenar por rating
df_movies_sorted = df_movies.sort_values(by="rating", ascending=False)

# Filtrar > 9
top_movies = df_movies_sorted[df_movies_sorted["rating"] > 9.0]

display(top_movies.head())

# Exportar
try:
    df_movies.to_csv("movies.csv", index=False)
    df_series.to_csv("series.csv", index=False)

    df_movies.to_json("movies.json", orient="records", indent=4)
    df_series.to_json("series.json", orient="records", indent=4)

except Exception as e:
    print("Erro ao exportar:", e)


,id,title,year,rating,categoria
0,1,The Shawshank Redemption,1994,9.3,Obra-prima
1,2,The Godfather,1972,9.2,Obra-prima
2,3,The Dark Knight,2008,9.1,Obra-prima


AT-09

In [20]:
def classificar(nota):
    if nota >= 9.0:
        return "Obra-prima"
    elif nota >= 8.0:
        return "Excelente"
    elif nota >= 7.0:
        return "Bom"
    else:
        return "Mediano"

df_movies["categoria"] = df_movies["rating"].apply(classificar)

df_movies[["title", "rating", "categoria"]].head(10)


,title,rating,categoria
0,The Shawshank Redemption,9.3,Obra-prima
1,The Godfather,9.2,Obra-prima
2,The Dark Knight,9.1,Obra-prima
3,The Godfather Part II,9.0,Obra-prima
4,12 Angry Men,9.0,Obra-prima
5,The Lord of the Rings: The Return of the King,9.0,Obra-prima
6,Schindler's List,9.0,Obra-prima
7,The Lord of the Rings: The Fellowship of the Ring,8.9,Excelente
8,Pulp Fiction,8.8,Excelente
9,"The Good, the Bad and the Ugly",8.8,Excelente


AT-10

In [25]:
def classificar(nota):
    if nota >= 9.0:
        return "Obra-prima"
    elif nota >= 8.0:
        return "Excelente"
    elif nota >= 7.0:
        return "Bom"
    else:
        return "Mediano"

df_movies["categoria"] = df_movies["rating"].apply(classificar)

tabela_resumo = df_movies.pivot_table(
    index="categoria",
    columns="year",
    values="title",
    aggfunc="count",
    fill_value=0
)

display(tabela_resumo)

year,1946,1954,1957,1966,1972,1974,1975,1980,1990,1991,...,1994,1995,1998,1999,2001,2002,2003,2008,2010,2014
categoria,,,,,,,,,,,,,,,,,,,,,
Excelente,1,1,0,1,0,0,1,1,1,1,...,2,1,1,3,1,1,0,0,1,1
Obra-prima,0,0,1,0,1,1,0,0,0,0,...,1,0,0,0,0,0,1,1,0,0


AT-11


------------------------------------------
Estrutura:

project/
│── src/
│   ├── scraping.py
│   ├── models.py
│   ├── database.py
│   ├── analysis.py
│   ├── main.py
│── data/
│── config.json
│── requirements.txt
│── README.md
------------------------------------------
Arquivo config.json:

{
    "url": "https://m.imdb.com/chart/top/",
    "n_filmes": 250
}



AT-12

In [22]:
LINK_GITHUB = "https://github.com/SimoesDev1/AT-Python/blob/main/AT_Python.ipynb"